# Load a Toy Datasource

This tutorial shows how to load synthetic data using `ToyRaggedLoader` and inspect the resulting `SplitDatasetContainer`. The toy loader generates ragged (variable-length) sequences suitable for anomaly detection.

## Data flow

```
ToyRaggedLoader                    SplitDatasetContainer
      │                                    │
      │  load_data()                       │  features.train  → list of arrays (one per unit)
      │  split_data()                      │  features.val    → list of arrays
      │  get_data()  ─────────────────────►  features.test   → list of arrays
      │                                    │  target.train   → list of arrays
      │                                    │  target.val, target.test
```

In [ ]:
from picid.data.datasources.toy_example import ToyRaggedLoader

In [ ]:
loader = ToyRaggedLoader(data_dir=".", data_name="toy", task_mode="anomaly_detection")
loader.load_data()
loader.split_data()

In [ ]:
container = loader.get_data()


def _shape(a):
    return a.to_numpy().shape if hasattr(a, "to_numpy") else a.shape


train_feats = container.features.train
train_targs = container.target.train
print(f"Train units: {len(train_feats)}")
print(f"Feature shapes (first 3): {[_shape(a) for a in train_feats[:3]]}")
print(f"Target shapes (first 3): {[_shape(a) for a in train_targs[:3]]}")

## AbstractDataSourceLoader contract

Every datasource loader implements the `AbstractDataSourceLoader` interface:

- **`load_data()`** — Loads raw data and populates internal state.
- **`split_data()`** — Splits data into train/val/test (or logs a warning for predefined splits).
- **`get_data()`** — Returns a `SplitDatasetContainer`.

The container stores data by **type** (features, target, unit_id, metadata) and **split** (train, val, test). Each split value is a **list of arrays** — one array per unit. For ragged data, arrays may have different lengths along the time dimension.